## LLM Finetuning

In [1]:
"""
Fine-tuning Qwen for Medical Information Extraction
=====================================================
Model  : Qwen/Qwen2.5-1.5B-Instruct  (small, multilingual, Apache-2.0)
Task   : Structured extraction from clinical / medical free text
Method : QLoRA (4-bit NF4 quantisation + LoRA adapters via PEFT)
Trainer: Hugging Face TRL – SFTTrainer
 
Pip requirements (install once before running):
    pip install torch transformers datasets peft trl bitsandbytes accelerate
 
Optional but recommended:
    pip install wandb          # experiment tracking
    pip install sentencepiece  # some tokenisers need it
"""
import sys
print(sys.executable)


/home/tatiana.bladier/LLMs-for-Data-Extraction/myenv/bin/python


In [2]:
import trl, pkg_resources
print(pkg_resources.get_distribution("trl").version)

# Then find where it lives:
import subprocess
subprocess.run(["grep", "-r", "DataCollatorForCompletionOnlyLM", trl.__path__[0]])

/home/tatiana.bladier/LLMs-for-Data-Extraction/myenv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_660403/1104146690.py:1: FutureWarning: Support for Python 3.9 will be dropped in the next release (after its end-of-life on October 31, 2025). Please upgrade to Python 3.10 or newer.
  import trl, pkg_resources


0.24.0


CompletedProcess(args=['grep', '-r', 'DataCollatorForCompletionOnlyLM', '/home/tatiana.bladier/LLMs-for-Data-Extraction/myenv/lib64/python3.9/site-packages/trl'], returncode=1)

In [3]:
#import sys
#!{sys.executable} -m pip install torch
#!{sys.executable} -m pip install datasets
#!{sys.executable} -m pip install peft
#!{sys.executable} -m pip install trl
#!{sys.executable} -m pip install bitsandbytes>=0.39.0

In [4]:
import json
import os
from dataclasses import dataclass, field
from typing import Optional
 
import torch
from datasets import Dataset, DatasetDict
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from trl import SFTTrainer # DataCollatorForCompletionOnlyLM
#import transformers, pkg_resources
print(pkg_resources.get_distribution("transformers").version)

#from transformers import DataCollatorForCompletionOnlyLM 
 
 

4.57.6


In [5]:
import transformers
print(transformers.__version__)

4.57.6


In [6]:
# ── Inline replacement for DataCollatorForCompletionOnlyLM ───────────────────
# Removed from TRL ≥0.24 and not yet in all transformers builds.
# Masks every token before (and including) the response template so that
# the training loss is computed only on the assistant's reply.
class DataCollatorForCompletionOnlyLM:
    def __init__(self, response_template: str, tokenizer, ignore_index: int = -100):
        self.response_template = response_template
        self.tokenizer = tokenizer
        self.ignore_index = ignore_index
        # Pre-encode the template once (no special tokens)
        self.template_ids = tokenizer.encode(
            response_template, add_special_tokens=False
        )
 
    def __call__(self, features):
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        labels = batch["input_ids"].clone()
        tlen = len(self.template_ids)
 
        for i, seq in enumerate(labels):
            seq_list = seq.tolist()
            # Find the LAST occurrence of the template (assistant turn)
            mask_until = 0
            for j in range(len(seq_list) - tlen + 1):
                if seq_list[j: j + tlen] == self.template_ids:
                    mask_until = j + tlen  # mask prompt + template itself
            labels[i, :mask_until] = self.ignore_index
 
        batch["labels"] = labels
        return batch

In [7]:
# ── 1. Configuration ──────────────────────────────────────────────────────────
@dataclass
class Config:
    # ---- Model ---------------------------------------------------------------
    model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    # Alternative small Qwen multilingual options:
    #   "Qwen/Qwen2.5-0.5B-Instruct"   (~0.5 B params, fastest)
    #   "Qwen/Qwen2.5-3B-Instruct"     (~3  B params, better quality)
 
    # ---- LoRA ----------------------------------------------------------------
    lora_r: int = 16               # rank (8–64; higher = more capacity)
    lora_alpha: int = 32           # scaling factor (typically 2×r)
    lora_dropout: float = 0.05
    lora_target_modules: list = field(
        default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj",
                                 "gate_proj", "up_proj", "down_proj"]
    )
 
    # ---- Quantisation --------------------------------------------------------
    use_4bit: bool = True          # QLoRA; set False if you have >24 GB VRAM
    bnb_4bit_compute_dtype: str = "bfloat16"
    bnb_4bit_quant_type: str = "nf4"
    use_nested_quant: bool = True  # double quantisation
 
    # ---- Training ------------------------------------------------------------
    output_dir: str = "./qwen_medical_extraction"
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4   # effective batch = 8
    learning_rate: float = 2e-4
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    max_seq_length: int = 1024
    logging_steps: int = 10
    save_strategy: str = "epoch"
    fp16: bool = False
    bf16: bool = True              # use False on older GPUs that lack bf16
 
    # ---- Data ----------------------------------------------------------------
    val_size: float = 0.1          # fraction held-out for validation
    seed: int = 42
 
 
CFG = Config()

In [8]:
# ── 2. Toy Medical Dataset ────────────────────────────────────────────────────
# In production replace this with a real dataset, e.g.:
#   • i2b2 / n2c2 NLP challenges          (https://n2c2.dbmi.hms.harvard.edu/)
#   • MedMentions                          (https://github.com/chanzuckerberg/MedMentions)
#   • MIMIC-III discharge summaries        (requires credentialed access)
#   • datasets.load_dataset("bigbio/...")  (HF BigBIO wrappers)
 
RAW_EXAMPLES = [
    {
        "text": (
            "Patient: Maria Gonzalez, 54-year-old female. "
            "Chief complaint: chest pain radiating to the left arm for 2 hours. "
            "PMH: hypertension, type 2 diabetes mellitus. "
            "Medications: metformin 1000 mg twice daily, lisinopril 10 mg once daily. "
            "Vitals: BP 158/94, HR 102, SpO2 97%. "
            "Diagnosis: NSTEMI."
        ),
        "extraction": {
            "patient_name": "Maria Gonzalez",
            "age": 54,
            "sex": "female",
            "chief_complaint": "chest pain radiating to the left arm for 2 hours",
            "past_medical_history": ["hypertension", "type 2 diabetes mellitus"],
            "medications": [
                {"name": "metformin", "dose": "1000 mg", "frequency": "twice daily"},
                {"name": "lisinopril", "dose": "10 mg", "frequency": "once daily"},
            ],
            "vitals": {"BP": "158/94", "HR": 102, "SpO2": "97%"},
            "diagnosis": "NSTEMI",
        },
    },
    {
        "text": (
            "Patient: Jean-Pierre Dumont, male, 67 years old. "
            "Admitted for acute dyspnea and bilateral leg oedema. "
            "Known heart failure (EF 35%) and chronic kidney disease stage 3. "
            "Current treatment: furosemide 40 mg/day, carvedilol 25 mg twice daily, "
            "spironolactone 25 mg/day. "
            "Labs: creatinine 1.8 mg/dL, BNP 980 pg/mL. "
            "Impression: decompensated heart failure."
        ),
        "extraction": {
            "patient_name": "Jean-Pierre Dumont",
            "age": 67,
            "sex": "male",
            "chief_complaint": "acute dyspnea and bilateral leg oedema",
            "past_medical_history": [
                "heart failure (EF 35%)",
                "chronic kidney disease stage 3",
            ],
            "medications": [
                {"name": "furosemide", "dose": "40 mg", "frequency": "once daily"},
                {"name": "carvedilol", "dose": "25 mg", "frequency": "twice daily"},
                {"name": "spironolactone", "dose": "25 mg", "frequency": "once daily"},
            ],
            "labs": {"creatinine": "1.8 mg/dL", "BNP": "980 pg/mL"},
            "diagnosis": "decompensated heart failure",
        },
    },
    {
        "text": (
            "Patient: Yuki Tanaka, 29-year-old female, presented with rash, "
            "joint pain, and photosensitivity for 3 weeks. "
            "ANA positive (1:320), anti-dsDNA elevated. "
            "No prior significant medical history. "
            "Medications: ibuprofen 400 mg as needed. "
            "Impression: systemic lupus erythematosus (SLE)."
        ),
        "extraction": {
            "patient_name": "Yuki Tanaka",
            "age": 29,
            "sex": "female",
            "chief_complaint": "rash, joint pain, and photosensitivity for 3 weeks",
            "past_medical_history": [],
            "medications": [
                {"name": "ibuprofen", "dose": "400 mg", "frequency": "as needed"}
            ],
            "labs": {"ANA": "positive (1:320)", "anti-dsDNA": "elevated"},
            "diagnosis": "systemic lupus erythematosus (SLE)",
        },
    },
    # ── Add many more examples here for a real run ──
]
 

In [9]:
# ── 3. Prompt Template ────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a clinical NLP assistant. "
    "Given a medical text, extract structured information and return it as valid JSON. "
    "Include all fields present in the text; omit fields that are not mentioned. "
    "Always return ONLY the JSON object — no markdown, no explanation."
)
 
 
def build_chat_prompt(example: dict, tokenizer) -> str:
    """
    Build a single training string in Qwen's ChatML format.
    The model must learn to predict only the assistant turn (the JSON).
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": example["text"]},
        {
            "role": "assistant",
            "content": json.dumps(example["extraction"], ensure_ascii=False, indent=2),
        },
    ]
    # apply_chat_template handles the <|im_start|>/<|im_end|> tokens for Qwen
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

In [10]:
# ── 4. Dataset Preparation ────────────────────────────────────────────────────
 
def prepare_dataset(raw_examples: list, tokenizer, val_size: float, seed: int) -> DatasetDict:
    """Convert raw examples to a formatted HF DatasetDict."""
    formatted = [
        {"text": build_chat_prompt(ex, tokenizer)} for ex in raw_examples
    ]
    ds = Dataset.from_list(formatted)
    split = ds.train_test_split(test_size=val_size, seed=seed)
    return DatasetDict({"train": split["train"], "validation": split["test"]})

In [11]:
# ── 5. Model & Tokeniser Loading ──────────────────────────────────────────────
 
def load_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        padding_side="right",   # required for SFT with causal LMs
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer
 
 
def load_model(cfg: Config):
    compute_dtype = getattr(torch, cfg.bnb_4bit_compute_dtype)
 
    bnb_config = None
    if cfg.use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=cfg.use_nested_quant,
        )
 
    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        quantization_config=bnb_config,
        device_map="auto",          # spread across available GPUs / CPU
        trust_remote_code=True,
        torch_dtype=compute_dtype if not cfg.use_4bit else None,
    )
    model.config.use_cache = False  # disable KV-cache during training
    return model

def apply_lora(model, cfg: Config):
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=cfg.lora_target_modules,
        bias="none",
        inference_mode=False,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model
 

In [20]:
# ── 6. Training ───────────────────────────────────────────────────────────────
 
def train(cfg: Config):
    print("=" * 60)
    print(f"  Model : {cfg.model_name}")
    print(f"  QLoRA : {cfg.use_4bit}  |  LoRA rank : {cfg.lora_r}")
    print("=" * 60)

    # 6-a. Tokeniser
    tokenizer = load_tokenizer(cfg.model_name)

    # 6-b. Dataset
    dataset = prepare_dataset(RAW_EXAMPLES, tokenizer, cfg.val_size, cfg.seed)
    print(f"\nDataset — train: {len(dataset['train'])}, val: {len(dataset['validation'])}\n")

    # 6-c. Model + LoRA
    model = load_model(cfg)
    model = apply_lora(model, cfg)

    # 6-d. Response-only data collator
    response_template = "<|im_start|>assistant\n"
    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template,
        tokenizer=tokenizer,
    )

    # 6-e. SFTConfig (replaces TrainingArguments + SFTTrainer kwargs in TRL>=0.9)
    from trl import SFTConfig
    training_args = SFTConfig(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.num_train_epochs,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        lr_scheduler_type=cfg.lr_scheduler_type,
        warmup_ratio=cfg.warmup_ratio,
        weight_decay=cfg.weight_decay,
        fp16=cfg.fp16,
        bf16=cfg.bf16,
        logging_steps=cfg.logging_steps,
        save_strategy=cfg.save_strategy,
        eval_strategy=cfg.save_strategy,
        load_best_model_at_end=True,
        report_to="none",
        seed=cfg.seed,
        dataset_text_field="text",
        max_length=cfg.max_seq_length,
    )

    # 6-f. SFTTrainer — `tokenizer` renamed to `processing_class` in TRL>=0.24
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=collator,
        args=training_args,
    )

    # 6-g. Train!
    print("\nStarting training …\n")
    trainer.train()

    # 6-h. Save final adapter + tokeniser
    final_path = os.path.join(cfg.output_dir, "final_adapter")
    trainer.model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    print(f"\n✅ Adapter saved to: {final_path}")

    return trainer, tokenizer



In [21]:
# ── 7. Inference Helper ───────────────────────────────────────────────────────
 
def extract_medical_info(
    text: str,
    model,
    tokenizer,
    max_new_tokens: int = 512,
    device: str = "cuda",
) -> dict:
    """
    Run inference with the fine-tuned model and return a parsed JSON dict.
 
    Usage after training:
        from peft import PeftModel
        base = AutoModelForCausalLM.from_pretrained(CFG.model_name, ...)
        model = PeftModel.from_pretrained(base, "./qwen_medical_extraction/final_adapter")
        result = extract_medical_info(note, model, tokenizer)
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
 
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
 
    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
 
    try:
        return json.loads(raw_output)
    except json.JSONDecodeError:
        # Return raw string if model output isn't valid JSON yet
        return {"raw_output": raw_output}
 

In [22]:
# ── 8. Quick Smoke-Test (CPU, no training) ────────────────────────────────────
 
def smoke_test_prompt(tokenizer):
    """Verify the prompt template renders correctly — no GPU needed."""
    sample = RAW_EXAMPLES[0]
    rendered = build_chat_prompt(sample, tokenizer)
    print("\n── Rendered prompt (first example) ──────────────────────────\n")
    print(rendered)
    print("\n─────────────────────────────────────────────────────────────\n")
 

In [23]:
# ── 9. Entry Point ────────────────────────────────────────────────────────────
 
def configure(
    model: str = None,
    epochs: int = None,
    use_4bit: bool = None,
) -> Config:
    """
    Optionally override global CFG values and return the config object.
    Call this first if you want to change any setting before training.
 
    Example
    -------
    cfg = configure(model="Qwen/Qwen2.5-0.5B-Instruct", epochs=5)
    """
    if model is not None:
        CFG.model_name = model
    if epochs is not None:
        CFG.num_train_epochs = epochs
    if use_4bit is not None:
        CFG.use_4bit = use_4bit
 
    print("Active configuration:")
    for k, v in CFG.__dict__.items():
        print(f"  {k}: {v}")
    return CFG
 
 
def run_smoke_test(model: str = None) -> None:
    """
    Load the tokeniser and print a rendered prompt for the first example.
    No GPU or model weights needed — completes in a few seconds.
 
    Example
    -------
    run_smoke_test()
    run_smoke_test(model="Qwen/Qwen2.5-0.5B-Instruct")
    """
    model_name = model or CFG.model_name
    print(f"Loading tokeniser for: {model_name}")
    tok = load_tokenizer(model_name)
    smoke_test_prompt(tok)
 
 
def run_training(cfg: Config = None):
    cfg = cfg or CFG
    trainer, tokenizer = train(cfg)
    return trainer, tokenizer
 
 
def run_inference(
    text: str,
    model,
    tokenizer,
    max_new_tokens: int = 512,
) -> dict:
    """
    Extract structured information from a free-text medical note.
    Returns a dict (parsed JSON) or {"raw_output": "..."} if parsing fails.
 
    Example
    -------
    note = "Patient: Anna Müller, 45F. Diagnosis: migraine. Medication: ibuprofen 400 mg prn."
    result = run_inference(note, model, tokenizer)
    print(result)
    """
    device = next(model.parameters()).device
    result = extract_medical_info(text, model, tokenizer, max_new_tokens, device)
    print(json.dumps(result, indent=2, ensure_ascii=False))
    return result
 
 
def load_finetuned(adapter_path: str = None, cfg: Config = None):
    """
    Load the base model and merge the saved LoRA adapter for inference.
    Uses the output directory from CFG if no path is provided.
 
    Returns (model, tokenizer).
 
    Example
    -------
    model, tokenizer = load_finetuned()
    # or with an explicit path:
    model, tokenizer = load_finetuned("./my_run/final_adapter")
    """
    from peft import PeftModel
 
    cfg = cfg or CFG
    adapter_path = adapter_path or os.path.join(cfg.output_dir, "final_adapter")
 
    print(f"Loading base model : {cfg.model_name}")
    print(f"Loading adapter    : {adapter_path}")
 
    tokenizer = load_tokenizer(cfg.model_name)
    base_model = load_model(cfg)
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
 
    print("\u2705 Model ready for inference.")
    return model, tokenizer

In [24]:
"""
# Install dependencies
pip install torch transformers datasets peft trl bitsandbytes accelerate

# Verify prompt rendering (no GPU needed)
python finetune_medical_qwen.py --smoke-test

# Full training run
python finetune_medical_qwen.py --epochs 3

# Without 4-bit quantisation (if you have ≥24 GB VRAM)
python finetune_medical_qwen.py --no-4bit
"""

'\n# Install dependencies\npip install torch transformers datasets peft trl bitsandbytes accelerate\n\n# Verify prompt rendering (no GPU needed)\npython finetune_medical_qwen.py --smoke-test\n\n# Full training run\npython finetune_medical_qwen.py --epochs 3\n\n# Without 4-bit quantisation (if you have ≥24 GB VRAM)\npython finetune_medical_qwen.py --no-4bit\n'

In [25]:
cfg = configure(model="Qwen/Qwen2.5-0.5B-Instruct", epochs=1, use_4bit=True)


Active configuration:
  model_name: Qwen/Qwen2.5-0.5B-Instruct
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  lora_target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  use_4bit: True
  bnb_4bit_compute_dtype: bfloat16
  bnb_4bit_quant_type: nf4
  use_nested_quant: True
  output_dir: ./qwen_medical_extraction
  num_train_epochs: 1
  per_device_train_batch_size: 2
  gradient_accumulation_steps: 4
  learning_rate: 0.0002
  lr_scheduler_type: cosine
  warmup_ratio: 0.05
  weight_decay: 0.01
  max_seq_length: 1024
  logging_steps: 10
  save_strategy: epoch
  fp16: False
  bf16: True
  val_size: 0.1
  seed: 42


In [26]:
run_smoke_test()


Loading tokeniser for: Qwen/Qwen2.5-0.5B-Instruct

── Rendered prompt (first example) ──────────────────────────

<|im_start|>system
You are a clinical NLP assistant. Given a medical text, extract structured information and return it as valid JSON. Include all fields present in the text; omit fields that are not mentioned. Always return ONLY the JSON object — no markdown, no explanation.<|im_end|>
<|im_start|>user
Patient: Maria Gonzalez, 54-year-old female. Chief complaint: chest pain radiating to the left arm for 2 hours. PMH: hypertension, type 2 diabetes mellitus. Medications: metformin 1000 mg twice daily, lisinopril 10 mg once daily. Vitals: BP 158/94, HR 102, SpO2 97%. Diagnosis: NSTEMI.<|im_end|>
<|im_start|>assistant
{
  "patient_name": "Maria Gonzalez",
  "age": 54,
  "sex": "female",
  "chief_complaint": "chest pain radiating to the left arm for 2 hours",
  "past_medical_history": [
    "hypertension",
    "type 2 diabetes mellitus"
  ],
  "medications": [
    {
      "name"

In [27]:
trainer, tokenizer = run_training()

  Model : Qwen/Qwen2.5-0.5B-Instruct
  QLoRA : True  |  LoRA rank : 16

Dataset — train: 2, val: 1

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Truncating eval dataset: 100%|██████████| 1/1 [00:00<00:00, 418.63 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting training …



You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,No log,0.655074,1.465256,769.000000,0.854305



✅ Adapter saved to: ./qwen_medical_extraction/final_adapter


In [30]:
note = "Patient: Anna Müller, 45F. Headache for 3 days. Medication: ibuprofen 400 mg prn."
result = run_inference(note, trainer.model, tokenizer)

RuntimeError: expected scalar type Float but found Half

In [29]:
#model, tokenizer = load_finetuned()          # uses CFG.output_dir automatically
model, tokenizer = load_finetuned("./qwen_medical_extraction/final_adapter")

Loading base model : Qwen/Qwen2.5-0.5B-Instruct
Loading adapter    : ./qwen_medical_extraction/final_adapter
✅ Model ready for inference.
